# Qwen2.5-Coder-7B-Instruct Download and Test
This notebook downloads the Qwen2.5-Coder-7B-Instruct model and tests it on a **test case generation** task — the exact task it will be used for in the autograder pipeline.

No HuggingFace login required — Qwen is not a gated model.

## Cell 1 — Install dependencies

In [1]:
!pip install -qU transformers huggingface_hub bitsandbytes accelerate

## Cell 2 — Check GPU

In [2]:
import torch

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM total:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1), 'GB')
    print('VRAM free:', round(torch.cuda.mem_get_info()[0] / 1024**3, 1), 'GB')
else:
    print('WARNING: No CUDA GPU detected. Model will run on CPU and will be very slow.')

CUDA available: True
GPU: NVIDIA RTX A4000
VRAM total: 16.0 GB
VRAM free: 14.9 GB


## Cell 3 — Download model (8-bit quantized)

In [3]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_name = "Qwen/Qwen2.5-Coder-7B-Instruct"

print("Downloading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_name)

print("Downloading model (8-bit quantized)...")
quantization_config = BitsAndBytesConfig(load_in_8bit=True)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    quantization_config=quantization_config,
)

print("\nModel loaded successfully!")
print("Device map:", model.hf_device_map if hasattr(model, 'hf_device_map') else "auto")

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]


Model loaded successfully!
Device map: auto


## Cell 4 — Test: Generate test cases from code
This mirrors the **exact task** used in the autograder's test generation module.

In [4]:
import torch
import json

messages = [
    {
        "role": "system",
        "content": "You are an expert software tester. Output ONLY valid JSON. No markdown, no backticks, no explanation."
    },
    {
        "role": "user",
        "content": """Generate test cases for the following:

QUESTION:
Write a function is_prime(n) that returns True if n is a prime number, else False.

SUBMITTED CODE:
def is_prime(n):
    n = int(input())
    if n < 2:
        print(f"{n} is not a prime number.")
    for i in range(2, int(n**0.5) + 1):
        if n % i == 0:
            print(f"{n} is not a prime number.")
            return
    print(f"{n} is a prime number.")

Generate exactly 5 test cases as a JSON array. Each item must have:
- "input": {{"n": value}}
- "expected_output": true or false
- "description": one sentence

Output only the JSON array."""
    }
]

formatted_prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(formatted_prompt, return_tensors="pt").to(model.device)

print("Generating test cases...")
with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=600,
        do_sample=False,
        temperature=None,
        top_p=None,
        pad_token_id=tokenizer.eos_token_id,
    )

new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
response = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

print("\n--- Raw Output ---")
print(response)

[transformers] The following generation flags are not valid and may be ignored: ['top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Generating test cases...


c:\Users\FAST\AppData\Local\anaconda3\Lib\site-packages\bitsandbytes\autograd\_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")



--- Raw Output ---
[
    {
        "input": {"n": 2},
        "expected_output": true,
        "description": "Check if 2 is a prime number."
    },
    {
        "input": {"n": 3},
        "expected_output": true,
        "description": "Check if 3 is a prime number."
    },
    {
        "input": {"n": 4},
        "expected_output": false,
        "description": "Check if 4 is a prime number."
    },
    {
        "input": {"n": 29},
        "expected_output": true,
        "description": "Check if 29 is a prime number."
    },
    {
        "input": {"n": 1},
        "expected_output": false,
        "description": "Check if 1 is a prime number."
    }
]


## Cell 5 — Validate JSON output
Check that the output is valid JSON and matches the expected schema.

In [5]:
import re

# Strip markdown code fences if present
clean = re.sub(r"```(?:json)?\s*", "", response).strip().rstrip("`")

try:
    test_cases = json.loads(clean)
    print(f"Valid JSON — {len(test_cases)} test cases generated\n")

    for i, tc in enumerate(test_cases, 1):
        has_input = "input" in tc and isinstance(tc["input"], dict)
        has_output = "expected_output" in tc
        has_desc = "description" in tc
        status = "OK" if (has_input and has_output and has_desc) else "MISSING FIELDS"
        print(f"  Test {i} [{status}]: {tc.get('description', 'no description')}")
        print(f"    input: {tc.get('input')}  |  expected: {tc.get('expected_output')}")

except json.JSONDecodeError as e:
    print(f"Invalid JSON: {e}")
    print("Raw output was:")
    print(clean)

Valid JSON — 5 test cases generated

  Test 1 [OK]: Check if 2 is a prime number.
    input: {'n': 2}  |  expected: True
  Test 2 [OK]: Check if 3 is a prime number.
    input: {'n': 3}  |  expected: True
  Test 3 [OK]: Check if 4 is a prime number.
    input: {'n': 4}  |  expected: False
  Test 4 [OK]: Check if 29 is a prime number.
    input: {'n': 29}  |  expected: True
  Test 5 [OK]: Check if 1 is a prime number.
    input: {'n': 1}  |  expected: False


## Cell 6 — Check VRAM usage after loading

In [6]:
if torch.cuda.is_available():
    used = torch.cuda.memory_allocated(0) / 1024**3
    total = torch.cuda.get_device_properties(0).total_memory / 1024**3
    free = torch.cuda.mem_get_info()[0] / 1024**3
    print(f"VRAM used:  {used:.1f} GB")
    print(f"VRAM free:  {free:.1f} GB")
    print(f"VRAM total: {total:.1f} GB")

VRAM used:  8.1 GB
VRAM free:  6.4 GB
VRAM total: 16.0 GB
